# Notebook 15 - Portfolio Workflow Review and Case Study

## Purpose

Notebook 15 is the Milestone 18 portfolio workflow review and case-study
notebook for `christophermoverton/fintech-stratlake-notebook-workflows`.
It is a reviewer-facing entry point that connects the imported Notebook 00-14
sequence into one source-safe Fintech + StratLake workflow story.

## Repository Role

This notebook belongs to the notebook workflow repository. It orchestrates and
reviews native upstream behavior; it does not replace `fintech-market-ingestion`
or `stratlake-trade-engine`.

## Expected Runtime

- Runtime: Google Colab or a prepared local notebook runtime.
- Active workspace: `/content` in Colab.
- Persistence/archive storage: Google Drive or an operator-reviewed local
  archive root.
- Runtime secrets: provide values only through runtime secret mechanisms; no secret values are committed.

## Upstream Apps

- `fintech-market-ingestion`: market-data ingestion, session setup, archive
  backup, archive validation, archive inspection, and archive restore.
- `stratlake-trade-engine`: notebook/session setup, feature generation,
  strategy execution, strategy comparison, campaign/portfolio-associated
  execution, artifacts, evidence review, governance observation, and archive
  checkpoint/restore surfaces.

## Commit Safety

Committed source must remain output-free, execution-count-null,
metadata-minimized, preview-default, and free of secret values, tokens, real
Drive paths, local paths, runtime identifiers, generated artifacts, logs, and
executed outputs.


## 2. Portfolio Case-Study Thesis and Reviewer Reading Paths

Notebook 15 presents the repository as a portfolio-grade workflow
orchestration layer: Fintech ingests or restores market data, StratLake
consumes the curated handoff to generate or restore features, native StratLake
strategy and campaign commands create research artifacts, and downstream
review surfaces inspect those artifacts with explicit evidence and non-claim
boundaries.

| Reviewer path | Suggested reading |
|---|---|
| Quick portfolio review | Read this notebook top to bottom in preview mode. |
| Data-platform review | Focus on Fintech ingestion/restore, handoff roots, and StratLake feature sections. |
| Research-workflow review | Focus on strategy selection, native execution, comparison, and caveat sections. |
| Evidence-boundary review | Focus on artifact review, evidence-pack, governance, and non-claim sections. |
| Future maintainer review | Follow the workflow map and the M18 follow-up issue handoff. |

This notebook demonstrates notebook-first workflow design, native-command-first
integration, archive/restore thinking, and reviewer-oriented source hygiene.
It does not demonstrate investment quality, alpha, strategy approval,
statistical significance, promotion readiness, governance readiness,
production readiness, deployment readiness, live-trading suitability, or
source/runtime equivalence.


## 3. Source-Safe Runtime Profile Selector

The committed default is `portfolio_preview`. It performs no install, Drive
mount, restore, ingestion, feature generation, strategy execution, portfolio
execution, archive checkpoint, or write behavior.

Runtime profiles are intentionally explicit. Fresh Fintech ingestion and
Fintech archive restore are mutually exclusive for the same market-data
handoff. Fresh StratLake feature generation and StratLake feature archive
restore are mutually exclusive for the same feature handoff. Strategy
execution and strategy artifact restoration are separate paths. Portfolio or
campaign execution is explicitly gated.


In [ ]:
import json
import os
import shlex
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

try:
    from IPython.display import Markdown, display
except Exception:
    Markdown = None
    display = None

try:
    import pandas as pd
except Exception:
    pd = None

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    drive = None
    IN_COLAB = False


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def env_true(name: str, default: bool = False) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "on"}


def display_markdown(text: str) -> None:
    if display is not None and Markdown is not None:
        display(Markdown(text))
    else:
        print(text)


def display_rows(rows: list[dict[str, Any]], *, max_rows: int = 20) -> None:
    shown = rows[:max_rows]
    if pd is not None and display is not None:
        display(pd.DataFrame(shown))
    else:
        for row in shown:
            print(row)


def command_text(command: list[str]) -> str:
    return " ".join(shlex.quote(str(part)) for part in command)


def command_available(command: str) -> bool:
    return shutil.which(command) is not None


CAVEATS: list[str] = []
COMMAND_RESULTS: list[dict[str, Any]] = []


In [ ]:
NOTEBOOK15_PROFILE = os.environ.get(
    "NOTEBOOK15_PROFILE",
    "portfolio_preview"
).strip() or "portfolio_preview"

VALID_NOTEBOOK15_PROFILES = {
    "portfolio_preview",
    "fintech_market_data_ingestion_run",
    "fintech_market_data_archive_restore",
    "stratlake_feature_generation_run",
    "stratlake_feature_archive_restore",
    "strategy_execution_run",
    "strategy_artifact_restore_review",
    "portfolio_execution_run",
    "workflow_archive_checkpoint",
    "workflow_archive_restore",
}

if NOTEBOOK15_PROFILE not in VALID_NOTEBOOK15_PROFILES:
    raise ValueError(f"Unsupported NOTEBOOK15_PROFILE: {NOTEBOOK15_PROFILE!r}")

# Profile gates describe the workflow branch under test. They are visible during
# cold-smoke runs even when native command execution is not confirmed.
ALLOW_INSTALL = NOTEBOOK15_PROFILE != "portfolio_preview"
ALLOW_DRIVE_MOUNT = NOTEBOOK15_PROFILE != "portfolio_preview"
ALLOW_FINTECH_INIT = NOTEBOOK15_PROFILE in {
    "fintech_market_data_ingestion_run",
    "fintech_market_data_archive_restore",
    "stratlake_feature_generation_run",
}
ALLOW_STRATLAKE_INIT = NOTEBOOK15_PROFILE in {
    "stratlake_feature_generation_run",
    "stratlake_feature_archive_restore",
    "strategy_execution_run",
    "portfolio_execution_run",
}

ALLOW_FINTECH_INGESTION = NOTEBOOK15_PROFILE == "fintech_market_data_ingestion_run"
ALLOW_FINTECH_ARCHIVE_RESTORE = NOTEBOOK15_PROFILE == "fintech_market_data_archive_restore"
ALLOW_FEATURE_GENERATION = NOTEBOOK15_PROFILE == "stratlake_feature_generation_run"
ALLOW_FEATURE_ARCHIVE_RESTORE = NOTEBOOK15_PROFILE == "stratlake_feature_archive_restore"
ALLOW_STRATEGY_EXECUTION = NOTEBOOK15_PROFILE == "strategy_execution_run"
ALLOW_STRATEGY_ARTIFACT_RESTORE = NOTEBOOK15_PROFILE == "strategy_artifact_restore_review"
ALLOW_PORTFOLIO_EXECUTION = NOTEBOOK15_PROFILE == "portfolio_execution_run"
ALLOW_WORKFLOW_ARCHIVE_CHECKPOINT = NOTEBOOK15_PROFILE == "workflow_archive_checkpoint"
ALLOW_WORKFLOW_ARCHIVE_RESTORE = NOTEBOOK15_PROFILE == "workflow_archive_restore"

# Native execution is a separate manual confirmation. A selected runtime profile
# can activate its branch while still recording commands as skipped previews.
ALLOW_NATIVE_COMMAND_EXECUTION = os.environ.get(
    "NOTEBOOK15_ALLOW_NATIVE_COMMAND_EXECUTION",
    "0"
).strip() == "1"

EXPECTED_GATES_BY_PROFILE = {
    "portfolio_preview": set(),
    "fintech_market_data_ingestion_run": {"ALLOW_FINTECH_INGESTION"},
    "fintech_market_data_archive_restore": {"ALLOW_FINTECH_ARCHIVE_RESTORE"},
    "stratlake_feature_generation_run": {"ALLOW_FEATURE_GENERATION"},
    "stratlake_feature_archive_restore": {"ALLOW_FEATURE_ARCHIVE_RESTORE"},
    "strategy_execution_run": {"ALLOW_STRATEGY_EXECUTION"},
    "strategy_artifact_restore_review": {"ALLOW_STRATEGY_ARTIFACT_RESTORE"},
    "portfolio_execution_run": {"ALLOW_PORTFOLIO_EXECUTION"},
    "workflow_archive_checkpoint": {"ALLOW_WORKFLOW_ARCHIVE_CHECKPOINT"},
    "workflow_archive_restore": {"ALLOW_WORKFLOW_ARCHIVE_RESTORE"},
}

GATE_VALUES = {
    "ALLOW_FINTECH_INGESTION": ALLOW_FINTECH_INGESTION,
    "ALLOW_FINTECH_ARCHIVE_RESTORE": ALLOW_FINTECH_ARCHIVE_RESTORE,
    "ALLOW_FEATURE_GENERATION": ALLOW_FEATURE_GENERATION,
    "ALLOW_FEATURE_ARCHIVE_RESTORE": ALLOW_FEATURE_ARCHIVE_RESTORE,
    "ALLOW_STRATEGY_EXECUTION": ALLOW_STRATEGY_EXECUTION,
    "ALLOW_STRATEGY_ARTIFACT_RESTORE": ALLOW_STRATEGY_ARTIFACT_RESTORE,
    "ALLOW_PORTFOLIO_EXECUTION": ALLOW_PORTFOLIO_EXECUTION,
    "ALLOW_WORKFLOW_ARCHIVE_CHECKPOINT": ALLOW_WORKFLOW_ARCHIVE_CHECKPOINT,
    "ALLOW_WORKFLOW_ARCHIVE_RESTORE": ALLOW_WORKFLOW_ARCHIVE_RESTORE,
}

expected_enabled_gates = EXPECTED_GATES_BY_PROFILE[NOTEBOOK15_PROFILE]
actual_enabled_gates = {
    gate_name for gate_name, gate_value in GATE_VALUES.items() if gate_value
}

if actual_enabled_gates != expected_enabled_gates:
    raise RuntimeError(
        "Notebook 15 profile gate mismatch: "
        f"profile={NOTEBOOK15_PROFILE!r}, "
        f"expected_enabled_gates={sorted(expected_enabled_gates)!r}, "
        f"actual_enabled_gates={sorted(actual_enabled_gates)!r}"
    )

if ALLOW_FINTECH_INGESTION and ALLOW_FINTECH_ARCHIVE_RESTORE:
    raise RuntimeError(
        "Invalid Notebook 15 profile state: Fintech ingestion and Fintech archive "
        "restore cannot both be enabled for the same market-data handoff."
    )

if ALLOW_FEATURE_GENERATION and ALLOW_FEATURE_ARCHIVE_RESTORE:
    raise RuntimeError(
        "Invalid Notebook 15 profile state: StratLake feature generation and "
        "feature archive restore cannot both be enabled for the same feature state."
    )

if ALLOW_STRATEGY_EXECUTION and ALLOW_STRATEGY_ARTIFACT_RESTORE:
    raise RuntimeError(
        "Invalid Notebook 15 profile state: strategy execution and strategy "
        "artifact restore cannot both be enabled for the same strategy state."
    )

profile_branch_status = (
    "Preview profile selected; no workflow branch is active."
    if not actual_enabled_gates
    else (
        "Profile branch activated with native command execution confirmed."
        if ALLOW_NATIVE_COMMAND_EXECUTION
        else "Profile branch activated, native command execution not confirmed; commands were recorded as skipped previews."
    )
)

profile_summary = {
    "selected_profile": NOTEBOOK15_PROFILE,
    "source_safe_committed_default": NOTEBOOK15_PROFILE == "portfolio_preview",
    "enabled_profile_gates": sorted(actual_enabled_gates),
    "native_command_execution_confirmed": ALLOW_NATIVE_COMMAND_EXECUTION,
    "profile_branch_status": profile_branch_status,
    "install_profile_requested": ALLOW_INSTALL,
    "drive_mount_profile_requested": ALLOW_DRIVE_MOUNT,
}

print("Notebook 15 profile gate check passed.")
print(f"NOTEBOOK15_PROFILE = {NOTEBOOK15_PROFILE}")
print(f"Enabled profile gates = {sorted(actual_enabled_gates)}")
print(f"Native command execution confirmed = {ALLOW_NATIVE_COMMAND_EXECUTION}")
print(profile_branch_status)
profile_summary


In [ ]:
# Runtime-only override examples - keep commented in committed source.
#
# import os
#
# Fresh Fintech ingestion branch:
# os.environ["NOTEBOOK15_PROFILE"] = "fintech_market_data_ingestion_run"
# os.environ["NOTEBOOK15_SYMBOLS"] = "SPY,QQQ,IWM"
# os.environ["NOTEBOOK15_INGESTION_START"] = "<reviewed-start-date>"
# os.environ["NOTEBOOK15_INGESTION_END"] = "<reviewed-end-date>"
#
# StratLake feature generation branch after reviewed Fintech handoff:
# os.environ["NOTEBOOK15_PROFILE"] = "stratlake_feature_generation_run"
# os.environ["NOTEBOOK15_MARK_HANDOFF_USER_REVIEWED"] = "true"
#
# Strategy execution branch:
# os.environ["NOTEBOOK15_PROFILE"] = "strategy_execution_run"
# os.environ["NOTEBOOK15_SELECTED_STRATEGIES"] = "momentum_v1,mean_reversion_v1"
#
# Portfolio/campaign execution branch:
# os.environ["NOTEBOOK15_PROFILE"] = "portfolio_execution_run"
# os.environ["NOTEBOOK15_MARK_PORTFOLIO_INPUTS_USER_REVIEWED"] = "true"
#
# Archive checkpoint branch:
# os.environ["NOTEBOOK15_PROFILE"] = "workflow_archive_checkpoint"
#
# Native command execution remains separately confirmed. Leave this unset for
# cold-smoke branch checks so commands are recorded as skipped previews.
# os.environ["NOTEBOOK15_ALLOW_NATIVE_COMMAND_EXECUTION"] = "1"


## 4. Package Installation and Import Setup

Notebook 15 follows the Notebook 08-14 package-install precedent. Installation
is a live-runtime action and remains commented/inactive in committed source.
The preview profile does not install packages.


In [ ]:
# Optional dependency installation for an executed runtime copy.
# Keep inactive in committed/source-safe Notebook 15.
# !pip install -q "pandas-market-calendars>=5.0"
# !pip install -q --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ fintech-market-ingestion
# !pip install -q --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ stratlake-trade-engine

INSTALL_COMMANDS = [
    "pip install -q pandas-market-calendars>=5.0",
    "pip install -q --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ fintech-market-ingestion",
    "pip install -q --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ stratlake-trade-engine",
]

if ALLOW_INSTALL and ALLOW_NATIVE_COMMAND_EXECUTION:
    CAVEATS.append("Install branch enabled with native execution confirmed; package output must not be committed.")
elif ALLOW_INSTALL:
    CAVEATS.append("Install branch requested by profile, but native command execution is not confirmed; install commands remain skipped previews.")
else:
    CAVEATS.append("Package installation not run in preview/source-safe mode.")


## 5. Colab/Local Detection, Google Drive Guard, and Persistence Configuration

Active application work belongs under `/content` in Colab. Google Drive is
persistence, backup, archive, and restore storage only. Committed source uses
placeholder-only configuration and does not mount Drive in preview mode.


In [ ]:
WORKSPACE_ROOT = Path("/content") if IN_COLAB else Path.cwd()
DRIVE_FOLDER_NAME = os.environ.get("NOTEBOOK15_DRIVE_FOLDER_NAME", "REPLACE_WITH_DRIVE_FOLDER_NAME").strip()
DRIVE_ROOT = Path("<drive-root-placeholder>") if DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME" else Path("/content") / "drive" / "MyDrive" / DRIVE_FOLDER_NAME

FINTECH_ROOT = WORKSPACE_ROOT / "fintech-market-ingestion-demo"
STRATLAKE_ROOT = WORKSPACE_ROOT / "stratlake-trade-engine-demo"
MARKETLAKE_ROOT = FINTECH_ROOT / "data" / "curated"
FEATURES_DAILY_ROOT = STRATLAKE_ROOT / "data" / "curated" / "features_daily"

FINTECH_SESSION_NAME = "notebook15_fintech_portfolio_handoff"
STRATLAKE_SESSION_NAME = "notebook15_stratlake_portfolio_workflow"

DRIVE_MOUNT_CONFIRMED = ALLOW_DRIVE_MOUNT and ALLOW_NATIVE_COMMAND_EXECUTION
if DRIVE_MOUNT_CONFIRMED:
    if DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME":
        raise ValueError("Set NOTEBOOK15_DRIVE_FOLDER_NAME before mounting or writing Drive-backed state.")
    if IN_COLAB and drive is not None:
        drive.mount("/content/drive")
elif ALLOW_DRIVE_MOUNT:
    CAVEATS.append("Drive mount branch requested by profile, but native command execution is not confirmed; Drive remains unmounted.")
else:
    CAVEATS.append("Drive mount not run; Drive persistence remains runtime-only and gated.")

workspace_summary = [
    {"name": "WORKSPACE_ROOT", "value": WORKSPACE_ROOT.as_posix(), "classification": "active_runtime_workspace"},
    {"name": "DRIVE_ROOT", "value": "<placeholder>" if DRIVE_FOLDER_NAME == "REPLACE_WITH_DRIVE_FOLDER_NAME" else DRIVE_ROOT.as_posix(), "classification": "persistence_archive_restore_only"},
    {"name": "FINTECH_ROOT", "value": FINTECH_ROOT.as_posix(), "classification": "runtime_workspace_not_committed"},
    {"name": "STRATLAKE_ROOT", "value": STRATLAKE_ROOT.as_posix(), "classification": "runtime_workspace_not_committed"},
    {"name": "MARKETLAKE_ROOT", "value": MARKETLAKE_ROOT.as_posix(), "classification": "fintech_to_stratlake_handoff"},
    {"name": "NATIVE_EXECUTION_CONFIRMED", "value": str(ALLOW_NATIVE_COMMAND_EXECUTION), "classification": "second_level_runtime_confirmation"},
]
display_rows(workspace_summary)


## 6. Fintech Workspace/Session Initialization or Preview

Fintech initialization uses the established `fintech-init-project` command
shape. The notebook previews the command unless the compatible profile and
explicit allow gate are both enabled.


In [ ]:
def run_command(command: list[str], *, cwd: Path | None = None, allow_run: bool = False, label: str = "") -> dict[str, Any]:
    profile_gate_active = bool(allow_run)
    should_execute = profile_gate_active and ALLOW_NATIVE_COMMAND_EXECUTION
    row = {
        "label": label or command[0],
        "command": command_text(command),
        "cwd": cwd.as_posix() if cwd else None,
        "profile_gate_active": profile_gate_active,
        "native_command_execution_confirmed": ALLOW_NATIVE_COMMAND_EXECUTION,
        "allow_run": should_execute,
        "skipped": not should_execute,
        "skipped_preview": profile_gate_active and not ALLOW_NATIVE_COMMAND_EXECUTION,
        "returncode": None,
        "stdout_tail": "",
        "stderr_tail": "",
        "started_at": utc_now_iso(),
    }
    if not should_execute:
        COMMAND_RESULTS.append(row)
        return row
    started = time.time()
    completed = subprocess.run(command, cwd=str(cwd) if cwd else None, text=True, capture_output=True, check=False)
    row.update({
        "skipped": False,
        "skipped_preview": False,
        "returncode": completed.returncode,
        "stdout_tail": completed.stdout[-2000:],
        "stderr_tail": completed.stderr[-2000:],
        "duration_seconds": round(time.time() - started, 3),
    })
    COMMAND_RESULTS.append(row)
    return row


fintech_init_cmd = [
    "fintech-init-project",
    "--root", FINTECH_ROOT.as_posix(),
    "--session-name", FINTECH_SESSION_NAME,
    "--with-session",
    "--colab-profile",
]
fintech_init_result = run_command(fintech_init_cmd, allow_run=ALLOW_FINTECH_INIT, label="fintech init project")
fintech_init_result


## 7. Fintech Market-Data Workflow: Optional Live Ingestion or Archive Restore

Notebook 15 supports two Fintech market-data paths:

- Fresh ingestion through `fintech-backfill-daily`.
- Archive restore through `fintech-backup-data restore`.

They are mutually exclusive for the same handoff state and disabled by default.


In [ ]:
NOTEBOOK15_SYMBOLS = os.environ.get("NOTEBOOK15_SYMBOLS", "SPY,QQQ,IWM")
INGESTION_START = os.environ.get("NOTEBOOK15_INGESTION_START", "<reviewed-start-date>")
INGESTION_END = os.environ.get("NOTEBOOK15_INGESTION_END", "<reviewed-end-date>")
DAILY_BARS_ROOT = MARKETLAKE_ROOT / "bars_daily"

fintech_ingestion_cmd = [
    "fintech-backfill-daily",
    "--symbols", NOTEBOOK15_SYMBOLS,
    "--start", INGESTION_START,
    "--end", INGESTION_END,
    "--out", DAILY_BARS_ROOT.as_posix(),
    "--feed", os.environ.get("NOTEBOOK15_DATA_FEED", "iex"),
    "--source", os.environ.get("NOTEBOOK15_DATA_SOURCE", "alpaca_iex"),
    "--window", os.environ.get("NOTEBOOK15_BACKFILL_WINDOW", "month"),
]

fintech_restore_cmd = [
    "fintech-backup-data",
    "restore",
    "--backup-pack-dir", os.environ.get("NOTEBOOK15_FINTECH_BACKUP_PACK_DIR", "<reviewed-fintech-backup-pack-dir>"),
    "--restore-root", FINTECH_ROOT.as_posix(),
    "--overwrite-policy", os.environ.get("NOTEBOOK15_FINTECH_RESTORE_OVERWRITE_POLICY", "fail"),
]

fintech_ingestion_result = run_command(fintech_ingestion_cmd, allow_run=ALLOW_FINTECH_INGESTION, label="fintech daily bars ingestion")
fintech_restore_result = run_command(fintech_restore_cmd, allow_run=ALLOW_FINTECH_ARCHIVE_RESTORE, label="fintech market-data archive restore")

display_rows([
    {"path": "fresh_ingestion", "enabled": ALLOW_FINTECH_INGESTION, "command": command_text(fintech_ingestion_cmd)},
    {"path": "archive_restore", "enabled": ALLOW_FINTECH_ARCHIVE_RESTORE, "command": command_text(fintech_restore_cmd)},
])


## 8. Fintech Data Inspection and Handoff Root Validation

This section reviews the Fintech-to-StratLake handoff root without loading or
committing generated market data. Runtime observations are display-only.


In [ ]:
def inspect_path(path: Path, classification: str) -> dict[str, Any]:
    return {
        "path": path.as_posix(),
        "exists": path.exists(),
        "is_dir": path.is_dir(),
        "classification": classification,
    }


handoff_rows = [
    inspect_path(MARKETLAKE_ROOT, "fintech_curated_handoff_root"),
    inspect_path(DAILY_BARS_ROOT, "fintech_daily_bars_runtime_data"),
]

if not MARKETLAKE_ROOT.exists():
    CAVEATS.append("Fintech handoff root was not observed in this preview/runtime; feature generation remains blocked until reviewed input exists.")

display_rows(handoff_rows)


## 9. StratLake Workspace/Session Initialization

StratLake initialization uses the established `stratlake-init-session` pattern
with an explicit Fintech `--marketlake-root` handoff.


In [ ]:
stratlake_init_cmd = [
    "stratlake-init-session",
    "--root", STRATLAKE_ROOT.as_posix(),
    "--project-name", STRATLAKE_SESSION_NAME,
    "--marketlake-root", MARKETLAKE_ROOT.as_posix(),
    "--drive-root", DRIVE_ROOT.as_posix(),
    "--enable-drive-persistence",
    "--notebook-configs",
]

stratlake_init_result = run_command(stratlake_init_cmd, allow_run=ALLOW_STRATLAKE_INIT, label="stratlake init session")
stratlake_init_result


## 10. StratLake Feature Workflow: Optional Generation or Feature Archive Restore

Notebook 15 supports feature generation through `stratlake-build-features` or
feature/session restore through `stratlake-session-archive-restore-bootstrap`.
Both paths are source-safe previews unless explicitly enabled.


In [ ]:
FEATURE_START = os.environ.get("NOTEBOOK15_FEATURE_START", INGESTION_START)
FEATURE_END = os.environ.get("NOTEBOOK15_FEATURE_END", INGESTION_END)

feature_generation_cmd = [
    "stratlake-build-features",
    "--timeframe", os.environ.get("NOTEBOOK15_FEATURE_TIMEFRAME", "1D"),
    "--start", FEATURE_START,
    "--end", FEATURE_END,
    "--tickers", NOTEBOOK15_SYMBOLS,
    "--marketlake-root", MARKETLAKE_ROOT.as_posix(),
]

feature_restore_cmd = [
    "stratlake-session-archive-restore-bootstrap",
    "--archive-root", os.environ.get("NOTEBOOK15_STRATLAKE_ARCHIVE_ROOT", "<reviewed-stratlake-archive-root>"),
    "--target-root", STRATLAKE_ROOT.as_posix(),
    "--validate-before-restore",
    "--inspect-before-restore",
    "--overwrite-policy", os.environ.get("NOTEBOOK15_STRATLAKE_RESTORE_OVERWRITE_POLICY", "overwrite_allowed"),
]

feature_generation_result = run_command(feature_generation_cmd, cwd=STRATLAKE_ROOT, allow_run=ALLOW_FEATURE_GENERATION, label="stratlake feature generation")
feature_restore_result = run_command(feature_restore_cmd, cwd=STRATLAKE_ROOT, allow_run=ALLOW_FEATURE_ARCHIVE_RESTORE, label="stratlake feature archive restore")

display_rows([
    {"path": "fresh_feature_generation", "enabled": ALLOW_FEATURE_GENERATION, "command": command_text(feature_generation_cmd)},
    {"path": "feature_archive_restore", "enabled": ALLOW_FEATURE_ARCHIVE_RESTORE, "command": command_text(feature_restore_cmd)},
])


## 11. Feature-Root Validation and Fintech-to-StratLake Handoff Review

Feature validation is display-only. The notebook does not fabricate feature
data, repair feature files, or validate upstream contracts in place of
StratLake.


In [ ]:
feature_rows = [
    inspect_path(FEATURES_DAILY_ROOT, "native_stratlake_features_daily_root"),
    inspect_path(STRATLAKE_ROOT / "configs" / "strategies.yml", "native_strategy_catalog_candidate"),
    inspect_path(STRATLAKE_ROOT / "configs" / "portfolios.yml", "native_portfolio_catalog_candidate"),
]

if not FEATURES_DAILY_ROOT.exists():
    CAVEATS.append("No StratLake feature root observed; strategy and portfolio execution remain disabled unless restored/generated features are reviewed.")

display_rows(feature_rows)


## 12. Native Strategy Selection for at Least Two Supported Strategies

Notebook 15 uses native StratLake strategy names already present in the prior
notebook sequence where practical. Selection is configuration only; strategy
logic remains upstream-owned.


In [ ]:
SELECTED_STRATEGIES = [
    item.strip()
    for item in os.environ.get("NOTEBOOK15_SELECTED_STRATEGIES", "momentum_v1,mean_reversion_v1").split(",")
    if item.strip()
]

if len(SELECTED_STRATEGIES) < 2:
    raise ValueError("Notebook 15 requires at least two selected strategies for the portfolio case study.")

strategy_selection_rows = [
    {"strategy": strategy, "source": "native_stratlake_strategy_selection", "notebook_owned_logic": False}
    for strategy in SELECTED_STRATEGIES
]
display_rows(strategy_selection_rows)


## 13. Native Strategy Execution or Strategy Artifact Restoration

Strategy execution uses `stratlake-run-strategy`. Restored strategy artifacts
may be reviewed instead of fresh execution. The notebook does not implement
strategy, backtest, metrics, or artifact validation logic.


In [ ]:
STRATEGIES_CONFIG = STRATLAKE_ROOT / "configs" / "strategies.yml"

strategy_commands = [
    [
        "stratlake-run-strategy",
        "--strategies-config", STRATEGIES_CONFIG.as_posix(),
        "--strategy", strategy,
        "--start", FEATURE_START,
        "--end", FEATURE_END,
    ]
    for strategy in SELECTED_STRATEGIES
]

strategy_results = [
    run_command(command, cwd=STRATLAKE_ROOT, allow_run=ALLOW_STRATEGY_EXECUTION, label=f"strategy {command[4]}")
    for command in strategy_commands
]

strategy_artifact_review_root = Path(os.environ.get("NOTEBOOK15_STRATEGY_ARTIFACT_ROOT", "<reviewed-strategy-artifact-root>"))
if ALLOW_STRATEGY_ARTIFACT_RESTORE:
    CAVEATS.append("Strategy artifact restore/review enabled; treat restored artifacts as runtime evidence only.")
else:
    CAVEATS.append("Strategy artifact restoration not run.")

display_rows([
    {"strategy": strategy, "execution_enabled": ALLOW_STRATEGY_EXECUTION, "command": command_text(command)}
    for strategy, command in zip(SELECTED_STRATEGIES, strategy_commands)
])


## 14. Bounded Strategy Output Summary and Caveat Register

Runtime command summaries are bounded. Parsed or displayed rows are review aids,
not authoritative performance reporting.


In [ ]:
strategy_summary_rows = [
    {
        "strategy": strategy,
        "requested": ALLOW_STRATEGY_EXECUTION,
        "returncode": result.get("returncode"),
        "skipped": result.get("skipped"),
        "claim_boundary": "display_only_not_performance_claim",
    }
    for strategy, result in zip(SELECTED_STRATEGIES, strategy_results)
]

display_rows(strategy_summary_rows)


## 15. Portfolio/Campaign Case-Study Configuration

The conservative portfolio-associated case study uses the documented native
campaign surface from Notebook 13 when a portfolio-specific command is not
confirmed in this repository. Any future portfolio-specific command should be
documented before Notebook 15 promotes it from a gap to an executable surface.


In [ ]:
portfolio_case_study = {
    "case_study_name": "notebook15_portfolio_workflow_review",
    "selected_strategies": SELECTED_STRATEGIES,
    "native_surface": "stratlake-run-research-campaign",
    "portfolio_specific_command_confirmed": False,
    "portfolio_command_gap_follow_up": "Use documented StratLake portfolio command when confirmed; do not implement notebook-owned portfolio engine.",
    "claim_boundary": "workflow_demonstration_not_investment_or_performance_claim",
}

campaign_config_path = Path(os.environ.get("NOTEBOOK15_CAMPAIGN_CONFIG", "<reviewed-campaign-config>"))
portfolio_input_reviewed = env_true("NOTEBOOK15_MARK_PORTFOLIO_INPUTS_USER_REVIEWED")

portfolio_case_study


## 16. Native Portfolio/Campaign Execution or Artifact Restoration

Execution is gated and non-mutating by default. Notebook 15 uses native or
documented command surfaces only and can review restored artifacts instead of
fresh execution.


In [ ]:
campaign_execution_cmd = [
    "stratlake-run-research-campaign",
    "--config", campaign_config_path.as_posix(),
]

portfolio_input_reviewed = env_true("NOTEBOOK15_MARK_PORTFOLIO_INPUTS_USER_REVIEWED")
portfolio_execution_profile_ready = ALLOW_PORTFOLIO_EXECUTION and portfolio_input_reviewed
if ALLOW_PORTFOLIO_EXECUTION and not portfolio_input_reviewed:
    CAVEATS.append("Portfolio/campaign execution branch active but inputs were not explicitly reviewed; command remains a skipped preview.")

campaign_execution_result = run_command(
    campaign_execution_cmd,
    cwd=STRATLAKE_ROOT,
    allow_run=portfolio_execution_profile_ready,
    label="portfolio-associated native campaign execution",
)

campaign_artifact_root = Path(os.environ.get("NOTEBOOK15_CAMPAIGN_ARTIFACT_ROOT", "<reviewed-campaign-artifact-root>"))
campaign_execution_result


## 17. Bounded Portfolio Artifact Review and Display-Only Summaries

Artifact review is bounded and non-authoritative. Artifact presence does not
prove current-session execution, completeness, approval, readiness, or
investment suitability.


In [ ]:
expected_portfolio_artifacts = [
    "manifest.json",
    "run_registry.json",
    "metrics.json",
    "split_metrics.json",
    "promotion_gates.json",
    "report.md",
]

portfolio_artifact_rows = [
    {
        "expected_artifact": name,
        "root": "<placeholder>" if str(campaign_artifact_root).startswith("<") else campaign_artifact_root.as_posix(),
        "found": bool(campaign_artifact_root.exists() and (campaign_artifact_root / name).exists()),
        "classification": "candidate_native_artifact_display_only",
    }
    for name in expected_portfolio_artifacts
]

display_rows(portfolio_artifact_rows)


## 18. Archive Backup/Checkpoint and Restore Templates

Archive actions are source-safe templates until explicitly enabled. Generated
archive packs, restored files, checkpoint outputs, manifests, and logs must not
be committed.


In [ ]:
fintech_backup_cmd = [
    "fintech-backup-data", "pack",
    "--workspace-root", FINTECH_ROOT.as_posix(),
    "--source-dataset-root", DAILY_BARS_ROOT.as_posix(),
    "--backup-root", os.environ.get("NOTEBOOK15_FINTECH_BACKUP_ROOT", "<reviewed-fintech-backup-root>"),
    "--backup-id", os.environ.get("NOTEBOOK15_FINTECH_BACKUP_ID", "<reviewed-fintech-backup-id>"),
    "--shard-size-mb", os.environ.get("NOTEBOOK15_FINTECH_BACKUP_SHARD_SIZE_MB", "64"),
    "--dry-run",
]

stratlake_checkpoint_cmd = [
    "stratlake-session-archive-bootstrap",
    "--root", STRATLAKE_ROOT.as_posix(),
    "--archive-id", os.environ.get("NOTEBOOK15_STRATLAKE_ARCHIVE_ID", "<reviewed-stratlake-archive-id>"),
    "--output-root", os.environ.get("NOTEBOOK15_STRATLAKE_ARCHIVE_OUTPUT_ROOT", "<reviewed-stratlake-archive-output-root>"),
    "--include-features",
    "--include-artifacts",
    "--include-configs",
    "--validate-after-copy",
    "--inspect-after-copy",
]

fintech_backup_result = run_command(fintech_backup_cmd, allow_run=ALLOW_WORKFLOW_ARCHIVE_CHECKPOINT, label="fintech archive backup preview")
stratlake_checkpoint_result = run_command(stratlake_checkpoint_cmd, cwd=STRATLAKE_ROOT, allow_run=ALLOW_WORKFLOW_ARCHIVE_CHECKPOINT, label="stratlake archive checkpoint")

display_rows([
    {"surface": "fintech_backup", "profile_gate_active": ALLOW_WORKFLOW_ARCHIVE_CHECKPOINT, "native_execution_confirmed": ALLOW_NATIVE_COMMAND_EXECUTION, "command": command_text(fintech_backup_cmd)},
    {"surface": "stratlake_checkpoint", "profile_gate_active": ALLOW_WORKFLOW_ARCHIVE_CHECKPOINT, "native_execution_confirmed": ALLOW_NATIVE_COMMAND_EXECUTION, "command": command_text(stratlake_checkpoint_cmd)},
    {"surface": "fintech_restore", "profile_gate_active": ALLOW_FINTECH_ARCHIVE_RESTORE or ALLOW_WORKFLOW_ARCHIVE_RESTORE, "native_execution_confirmed": ALLOW_NATIVE_COMMAND_EXECUTION, "command": command_text(fintech_restore_cmd)},
    {"surface": "stratlake_restore", "profile_gate_active": ALLOW_FEATURE_ARCHIVE_RESTORE or ALLOW_WORKFLOW_ARCHIVE_RESTORE, "native_execution_confirmed": ALLOW_NATIVE_COMMAND_EXECUTION, "command": command_text(feature_restore_cmd)},
])


## 19. Evidence-Boundary Table

The notebook distinguishes committed source, optional runtime observations,
canonical upstream artifacts, derived review packs, governance observations,
and portfolio claims.


In [ ]:
evidence_boundaries = [
    {"surface": "committed_notebook_source", "authority": "repository source review", "allowed_claim": "source-safe structure", "forbidden_claim": "runtime success"},
    {"surface": "optional_runtime_observations", "authority": "manual runtime only", "allowed_claim": "observed command result when documented", "forbidden_claim": "source/runtime equivalence"},
    {"surface": "canonical_fintech_artifacts", "authority": "fintech-market-ingestion", "allowed_claim": "native artifact when generated/restored by native command", "forbidden_claim": "notebook-owned ingestion truth"},
    {"surface": "canonical_stratlake_artifacts", "authority": "stratlake-trade-engine", "allowed_claim": "native artifact candidate", "forbidden_claim": "notebook-owned validation truth"},
    {"surface": "derived_review_packs", "authority": "native evidence-review build plus validation", "allowed_claim": "non-authoritative derived review material", "forbidden_claim": "canonical governance evidence"},
    {"surface": "governance_observations", "authority": "native read-only governance commands", "allowed_claim": "bounded observation", "forbidden_claim": "promotion decision"},
    {"surface": "portfolio_claims", "authority": "none in committed source", "allowed_claim": "workflow demonstration", "forbidden_claim": "investment or performance recommendation"},
]

display_rows(evidence_boundaries)


## 20. Notebook 00-14 Workflow Map and Source-of-Record Links

Notebook 15 is the reviewer entry point. The detailed source of record remains
the prior notebooks, import audits, command-surface classifications, smoke
audit summaries, and merge-readiness documents.


In [ ]:
workflow_map = [
    {"notebook": "00", "role": "setup and storage overview", "detail_record": "docs/notebook_00_import_audit.md"},
    {"notebook": "01", "role": "Fintech daily bars extraction/backfill", "detail_record": "docs/notebook_01_import_audit.md"},
    {"notebook": "02", "role": "Fintech archive restore and session readiness", "detail_record": "docs/notebook_02_import_audit.md"},
    {"notebook": "03", "role": "Fintech archive backup pack and restore", "detail_record": "docs/notebook_03_import_audit.md"},
    {"notebook": "04", "role": "StratLake feature-series setup and dual-session bridge", "detail_record": "docs/notebook_04_import_audit.md"},
    {"notebook": "05", "role": "Fintech daily bars to StratLake feature generation", "detail_record": "docs/notebook_05_import_audit.md"},
    {"notebook": "06", "role": "Feature validation, archive, and handoff", "detail_record": "docs/notebook_06_import_audit.md"},
    {"notebook": "07", "role": "Feature consumption and baseline research", "detail_record": "docs/notebook_07_import_audit.md"},
    {"notebook": "08", "role": "Strategy backtest artifact review", "detail_record": "docs/notebook_08_import_audit.md"},
    {"notebook": "09", "role": "Strategy comparison and research review", "detail_record": "docs/notebook_09_import_audit.md"},
    {"notebook": "10", "role": "Walk-forward robustness and promotion review", "detail_record": "docs/notebook_10_import_audit.md"},
    {"notebook": "11", "role": "Expanded promotion evidence review", "detail_record": "docs/notebook_11_import_audit.md"},
    {"notebook": "12", "role": "Campaign evidence gap and promotion readiness review", "detail_record": "docs/notebook_12_import_audit.md"},
    {"notebook": "13", "role": "Native campaign execution and artifact generation", "detail_record": "docs/notebook_13_import_audit.md"},
    {"notebook": "14", "role": "Evidence review pack and governance observation", "detail_record": "docs/notebook_14_import_audit.md"},
]

display_rows(workflow_map, max_rows=30)


## 21. Final Portfolio Review Summary, Non-Goals, Validation Checklist, and Handoff

Notebook 15 is complete for M18.2 when it exists as clean source at
`notebooks/15_portfolio_workflow_review_and_case_study.ipynb`.

Validation checklist before commit:

- Notebook JSON is valid.
- Code-cell outputs are empty.
- Code-cell execution counts are `null`.
- Metadata is minimized.
- Default profile is `portfolio_preview`.
- Runtime examples are commented or gated.
- No secret values, tokens, Drive paths, local paths, runtime identifiers, logs,
  generated data, generated artifacts, or executed outputs are committed.

Follow-up mapping:

- #152 / M18.3: portfolio narrative and evidence-boundary guardrails.
- #153 / M18.4: source-only validation expansion for Notebook 15.
- #154 / M18.5: README, notebook index, and portfolio documentation links.
- #155 / M18.6: merge-readiness and handoff.


In [ ]:
NON_CLAIMS = [
    "investment_recommendation",
    "strategy_approval",
    "positive_alpha",
    "statistical_significance",
    "promotion_readiness",
    "governance_readiness",
    "deployment_readiness",
    "production_readiness",
    "live_trading_suitability",
    "source_runtime_equivalence",
    "portfolio_performance_quality",
    "authoritative_performance_reporting",
    "native_artifact_completeness_without_native_verification",
]

command_records_executed = sum(1 for row in COMMAND_RESULTS if not row.get("skipped"))
command_records_skipped = sum(1 for row in COMMAND_RESULTS if row.get("skipped"))
command_records_skipped_previews = sum(1 for row in COMMAND_RESULTS if row.get("skipped_preview"))

final_review_summary = {
    "notebook": "Notebook 15 - Portfolio Workflow Review and Case Study",
    "selected_profile": NOTEBOOK15_PROFILE,
    "enabled_profile_gates": sorted(actual_enabled_gates),
    "native_command_execution_confirmed": ALLOW_NATIVE_COMMAND_EXECUTION,
    "source_safe_committed_default": NOTEBOOK15_PROFILE == "portfolio_preview",
    "profile_branch_status": profile_branch_status,
    "command_records_total": len(COMMAND_RESULTS),
    "command_records_executed": command_records_executed,
    "command_records_skipped": command_records_skipped,
    "command_records_skipped_previews": command_records_skipped_previews,
    "selected_strategies": SELECTED_STRATEGIES,
    "portfolio_case_study": portfolio_case_study["case_study_name"],
    "caveat_count": len(CAVEATS),
    "caveats": CAVEATS,
    "non_claims": NON_CLAIMS,
    "final_stance": "notebook_15_portfolio_workflow_source_safe_review_ready",
}

final_review_summary
